# LLM & System Design Interviews

> **Target Roles**: ML Engineer, LLM Engineer, AI Research Engineer
>
> **Focus Areas**:
> - LLM Architecture Deep Dive
> - Production LLM Systems
> - Scaling & Optimization
> - RAG & Agent Systems
> - LLMOps & Monitoring
>
> **Format**: Technical deep dives + system design questions

## Section 1: LLM Architecture Questions

### Q1: Explain the GPT architecture in 2 minutes

**Answer Structure**:

**Core Idea**: Auto-regressive transformer that predicts next token.

**Components**:
1. **Token + Positional Embeddings**: Convert text to vectors
2. **N Transformer Blocks**: Each has:
   - Masked self-attention (causal)
   - Feed-forward network (2-layer MLP)
   - LayerNorm + residual connections
3. **Output**: Linear layer → softmax over vocabulary

**Key Features**:
- **Causal masking**: Token $i$ only sees tokens $< i$
- **Large scale**: GPT-3 has 175B params, 96 layers, 12,288 d_model
- **Pre-training**: Next token prediction on web text
- **Few-shot learning**: Task adaptation via prompting

**Math** (simplified):
$$P(x_t | x_{<t}) = \text{softmax}(W_o \cdot \text{Transformer}(x_{<t}))$$

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
import matplotlib.pyplot as plt
import numpy as np

# Minimal GPT architecture
class MinimalGPT(nn.Module):
    """Simplified GPT for interview explanation."""
    
    def __init__(self, vocab_size, d_model, n_layers, n_heads, max_len=1024):
        super().__init__()
        self.token_embedding = nn.Embedding(vocab_size, d_model)
        self.position_embedding = nn.Embedding(max_len, d_model)
        
        # Stack of transformer blocks
        self.blocks = nn.ModuleList([
            TransformerBlock(d_model, n_heads) 
            for _ in range(n_layers)
        ])
        
        self.ln_f = nn.LayerNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)
    
    def forward(self, x):
        B, T = x.shape
        
        # Embeddings
        tok_emb = self.token_embedding(x)  # (B, T, d_model)
        pos_emb = self.position_embedding(torch.arange(T, device=x.device))  # (T, d_model)
        x = tok_emb + pos_emb
        
        # Transformer blocks
        for block in self.blocks:
            x = block(x)
        
        # Output
        x = self.ln_f(x)
        logits = self.lm_head(x)  # (B, T, vocab_size)
        
        return logits

class TransformerBlock(nn.Module):
    def __init__(self, d_model, n_heads):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.attn = CausalSelfAttention(d_model, n_heads)
        self.ln2 = nn.LayerNorm(d_model)
        self.mlp = nn.Sequential(
            nn.Linear(d_model, 4 * d_model),
            nn.GELU(),
            nn.Linear(4 * d_model, d_model)
        )
    
    def forward(self, x):
        x = x + self.attn(self.ln1(x))  # Pre-LN + residual
        x = x + self.mlp(self.ln2(x))
        return x

class CausalSelfAttention(nn.Module):
    def __init__(self, d_model, n_heads):
        super().__init__()
        self.n_heads = n_heads
        self.d_k = d_model // n_heads
        
        self.qkv_proj = nn.Linear(d_model, 3 * d_model)
        self.out_proj = nn.Linear(d_model, d_model)
        
        # Causal mask (lower triangular)
        self.register_buffer('mask', torch.tril(torch.ones(1024, 1024)).view(1, 1, 1024, 1024))
    
    def forward(self, x):
        B, T, C = x.shape
        
        # Compute Q, K, V
        qkv = self.qkv_proj(x)
        q, k, v = qkv.split(C, dim=-1)
        
        # Reshape for multi-head
        q = q.view(B, T, self.n_heads, self.d_k).transpose(1, 2)
        k = k.view(B, T, self.n_heads, self.d_k).transpose(1, 2)
        v = v.view(B, T, self.n_heads, self.d_k).transpose(1, 2)
        
        # Attention
        attn = (q @ k.transpose(-2, -1)) / math.sqrt(self.d_k)
        attn = attn.masked_fill(self.mask[:, :, :T, :T] == 0, float('-inf'))
        attn = F.softmax(attn, dim=-1)
        
        out = attn @ v
        out = out.transpose(1, 2).contiguous().view(B, T, C)
        
        return self.out_proj(out)

# Example instantiation
model = MinimalGPT(vocab_size=50257, d_model=768, n_layers=12, n_heads=12)
print(f"GPT Parameters: {sum(p.numel() for p in model.parameters()) / 1e6:.1f}M")
print(f"\nArchitecture: {model}")

### Q2: What's the difference between GPT, BERT, and T5?

| Model | Architecture | Training | Inference | Use Cases |
|-------|--------------|----------|-----------|----------|
| **GPT** | Decoder-only | Causal LM (predict next token) | Left-to-right | Text generation, chat, code |
| **BERT** | Encoder-only | Masked LM (predict masked tokens) | Bidirectional | Classification, NER, Q&A |
| **T5** | Encoder-Decoder | Span corruption (seq2seq) | Both | Translation, summarization, Q&A |

**Key Insight**: 
- GPT for **generation** (causal)
- BERT for **understanding** (bidirectional)
- T5 for **transformation** (seq2seq)

**Modern trend**: Decoder-only (GPT-style) dominates due to scalability and in-context learning.

### Q3: Explain KV Cache optimization

**Problem**: During autoregressive generation, recomputing attention for all previous tokens is wasteful.

**Without KV Cache** (naive):
```
Step 1: Compute attention for token 1
Step 2: Compute attention for tokens 1, 2
Step 3: Compute attention for tokens 1, 2, 3
...
Step N: Compute attention for tokens 1...N
```
**Complexity**: $O(N^2)$ for sequence of length $N$

**With KV Cache**:
```
Step 1: Compute K1, V1, cache them
Step 2: Compute K2, V2, use cached K1, V1
Step 3: Compute K3, V3, use cached K1, K2, V1, V2
```
**Complexity**: $O(N)$ - linear!

**Implementation** (pseudo-code):

In [ ]:
class KVCacheAttention(nn.Module):
    """Attention with KV caching for fast generation."""
    
    def __init__(self, d_model, n_heads):
        super().__init__()
        self.n_heads = n_heads
        self.d_k = d_model // n_heads
        
        self.W_Q = nn.Linear(d_model, d_model)
        self.W_K = nn.Linear(d_model, d_model)
        self.W_V = nn.Linear(d_model, d_model)
        self.W_O = nn.Linear(d_model, d_model)
    
    def forward(self, x, past_kv=None, use_cache=False):
        """
        Args:
            x: Current token(s) (B, 1 or T, d_model)
            past_kv: Cached (K, V) from previous tokens
            use_cache: Whether to return updated cache
        """
        B, T, C = x.shape
        
        # Compute Q for current tokens
        Q = self.W_Q(x).view(B, T, self.n_heads, self.d_k).transpose(1, 2)
        
        # Compute K, V for current tokens
        K_new = self.W_K(x).view(B, T, self.n_heads, self.d_k).transpose(1, 2)
        V_new = self.W_V(x).view(B, T, self.n_heads, self.d_k).transpose(1, 2)
        
        # Concatenate with cached K, V
        if past_kv is not None:
            K_past, V_past = past_kv
            K = torch.cat([K_past, K_new], dim=2)  # (B, n_heads, T_total, d_k)
            V = torch.cat([V_past, V_new], dim=2)
        else:
            K, V = K_new, V_new
        
        # Attention
        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.d_k)
        attn = F.softmax(scores, dim=-1)
        out = torch.matmul(attn, V)
        
        # Reshape and project
        out = out.transpose(1, 2).contiguous().view(B, T, C)
        out = self.W_O(out)
        
        if use_cache:
            return out, (K, V)
        return out

# Memory Analysis
def kv_cache_memory(batch_size, seq_len, n_layers, d_model, n_heads):
    """Calculate KV cache memory usage."""
    d_k = d_model // n_heads
    
    # Per layer: 2 (K + V) × batch × n_heads × seq_len × d_k
    bytes_per_float16 = 2
    memory_per_layer = 2 * batch_size * n_heads * seq_len * d_k * bytes_per_float16
    total_memory = memory_per_layer * n_layers
    
    return total_memory / (1024 ** 3)  # Convert to GB

# Example: LLaMA-7B serving
config = {
    'batch_size': 8,
    'seq_len': 2048,
    'n_layers': 32,
    'd_model': 4096,
    'n_heads': 32
}

cache_mem = kv_cache_memory(**config)
print(f"KV Cache Memory for LLaMA-7B (batch=8, seq=2048): {cache_mem:.2f} GB")
print("\nOptimizations:")
print("1. PagedAttention (vLLM): Paged memory management")
print("2. Multi-Query Attention (MQA): Share K,V across heads")
print("3. Grouped-Query Attention (GQA): Middle ground (LLaMA 2)")
print("4. Quantization: INT8/INT4 KV cache")

## Section 2: Coding Challenges

### Challenge 1: Implement Beam Search

In [ ]:
def beam_search(model, input_ids, beam_width=5, max_length=50, temperature=1.0):
    """
    Beam search decoding for autoregressive LLM.
    
    Args:
        model: LLM with forward(input_ids) -> logits
        input_ids: Input token IDs (1, seq_len)
        beam_width: Number of beams
        max_length: Maximum generation length
        temperature: Sampling temperature
    
    Returns:
        best_sequence: Token IDs of best beam
        best_score: Log probability score
    """
    device = input_ids.device
    
    # Initialize beams: (sequence, score)
    beams = [(input_ids, 0.0)]
    
    for _ in range(max_length):
        all_candidates = []
        
        for seq, score in beams:
            # Get model predictions
            with torch.no_grad():
                logits = model(seq)[:, -1, :]  # (1, vocab_size)
            
            # Apply temperature
            logits = logits / temperature
            probs = F.softmax(logits, dim=-1)
            
            # Get top-k candidates
            top_probs, top_ids = torch.topk(probs, beam_width)
            
            for i in range(beam_width):
                candidate_seq = torch.cat([seq, top_ids[:, i:i+1]], dim=1)
                candidate_score = score + torch.log(top_probs[:, i]).item()
                all_candidates.append((candidate_seq, candidate_score))
        
        # Select top beam_width candidates
        beams = sorted(all_candidates, key=lambda x: x[1], reverse=True)[:beam_width]
        
        # Check if all beams ended with EOS
        if all(seq[0, -1].item() == 2 for seq, _ in beams):  # Assuming EOS token = 2
            break
    
    # Return best beam
    best_seq, best_score = beams[0]
    return best_seq, best_score

print("Beam Search Key Ideas:")
print("1. Maintain top-k hypotheses at each step")
print("2. Expand each hypothesis with top-k tokens")
print("3. Keep best k×k candidates")
print("4. Trade-off: k↑ → better quality but slower")
print("\nComplexity: O(k × L × V) where k=beam_width, L=length, V=vocab_size")

### Challenge 2: Implement Top-p (Nucleus) Sampling

In [ ]:
def nucleus_sampling(logits, p=0.9, temperature=1.0):
    """
    Top-p (nucleus) sampling.
    
    Sample from smallest set of tokens whose cumulative prob ≥ p.
    
    Args:
        logits: (batch, vocab_size)
        p: Nucleus probability threshold
        temperature: Sampling temperature
    
    Returns:
        sampled_token: (batch,)
    """
    # Apply temperature
    logits = logits / temperature
    probs = F.softmax(logits, dim=-1)
    
    # Sort probabilities in descending order
    sorted_probs, sorted_indices = torch.sort(probs, descending=True, dim=-1)
    
    # Compute cumulative probabilities
    cumulative_probs = torch.cumsum(sorted_probs, dim=-1)
    
    # Remove tokens with cumulative prob > p
    sorted_indices_to_remove = cumulative_probs > p
    
    # Keep at least 1 token (shift right)
    sorted_indices_to_remove[..., 1:] = sorted_indices_to_remove[..., :-1].clone()
    sorted_indices_to_remove[..., 0] = False
    
    # Zero out removed probabilities
    sorted_probs[sorted_indices_to_remove] = 0.0
    
    # Renormalize
    sorted_probs = sorted_probs / sorted_probs.sum(dim=-1, keepdim=True)
    
    # Sample from nucleus
    sampled_sorted_indices = torch.multinomial(sorted_probs, num_samples=1)
    sampled_token = sorted_indices.gather(-1, sampled_sorted_indices).squeeze(-1)
    
    return sampled_token

# Visualize sampling strategies
def compare_sampling_strategies():
    # Simulated logits (10 tokens)
    logits = torch.tensor([[2.0, 1.5, 1.0, 0.5, 0.2, 0.1, 0.05, 0.02, 0.01, 0.005]])
    probs = F.softmax(logits, dim=-1)
    
    print("Token Probabilities:")
    for i, p in enumerate(probs[0]):
        print(f"Token {i}: {p:.4f}")
    
    print("\n--- Sampling Comparison ---")
    
    # Greedy
    greedy = torch.argmax(probs, dim=-1)
    print(f"Greedy: Token {greedy.item()}")
    
    # Top-k (k=3)
    top_k_probs, top_k_ids = torch.topk(probs, k=3)
    print(f"Top-k (k=3): Tokens {top_k_ids[0].tolist()} with probs {top_k_probs[0].tolist()}")
    
    # Top-p (p=0.9)
    sorted_probs, _ = torch.sort(probs, descending=True, dim=-1)
    cumsum = torch.cumsum(sorted_probs, dim=-1)
    nucleus_size = (cumsum <= 0.9).sum().item() + 1
    print(f"Top-p (p=0.9): Nucleus size = {nucleus_size} tokens")
    
    print("\nWhen to use:")
    print("- Greedy: Deterministic, factual tasks (math, code)")
    print("- Top-k: Fixed diversity")
    print("- Top-p: Adaptive diversity (creative writing)")
    print("- Temperature: Controls randomness (0.7=focused, 1.5=creative)")

compare_sampling_strategies()

## Section 3: System Design Questions

### Q1: Design an LLM Serving System for 1M Daily Active Users

**Requirements**:
- Model: LLaMA-70B or GPT-3.5 equivalent
- Latency: P95 < 2s for 512 token generation
- Throughput: 10K concurrent users
- Cost: Optimize $/token

**Architecture**:

```
┌─────────────┐
│   Clients   │
└──────┬──────┘
       │
┌──────▼──────────────┐
│  API Gateway        │
│  - Rate limiting    │
│  - Auth             │
└──────┬──────────────┘
       │
┌──────▼──────────────┐
│  Load Balancer      │
└──────┬──────────────┘
       │
┌──────▼──────────────────────────────┐
│  Inference Servers (vLLM)           │
│  ┌────────────────────────────────┐ │
│  │  Request Queue                 │ │
│  │  ┌──────┬──────┬──────┬──────┐│ │
│  │  │ Req1 │ Req2 │ Req3 │ Req4 ││ │
│  │  └──────┴──────┴──────┴──────┘│ │
│  └────────────────────────────────┘ │
│  ┌────────────────────────────────┐ │
│  │  Continuous Batching           │ │
│  │  - Dynamic batch formation     │ │
│  │  - PagedAttention (KV cache)   │ │
│  └────────────────────────────────┘ │
│  ┌────────────────────────────────┐ │
│  │  GPU Pool (8× A100 80GB)       │ │
│  │  - Tensor Parallelism          │ │
│  │  - Pipeline Parallelism        │ │
│  └────────────────────────────────┘ │
└─────────────────────────────────────┘
       │
┌──────▼──────────────┐
│  Monitoring         │
│  - Latency metrics  │
│  - GPU utilization  │
│  - Cost tracking    │
└─────────────────────┘
```

**Key Components**:

1. **Inference Engine: vLLM**
   - PagedAttention for efficient KV cache
   - Continuous batching (process requests as they arrive)
   - 10-20x throughput improvement vs naive serving

2. **Model Parallelism**:
   - **Tensor Parallelism**: Split model layers across GPUs
   - **Pipeline Parallelism**: Different layers on different GPUs
   - For 70B model: 4-8× A100 80GB

3. **Optimizations**:
   - **Quantization**: FP16 or INT8 (2-4x speedup)
   - **Flash Attention**: Fused attention kernel
   - **Speculative Decoding**: Draft model + verification (2-3x speedup)

4. **Caching Strategy**:
   - **Prompt caching**: Cache KV for common prefixes (system prompts)
   - **Response caching**: Cache exact query matches
   - **Semantic caching**: Embed queries, cache similar ones

5. **Monitoring**:
   - **Latency**: P50, P95, P99 per request
   - **Throughput**: Tokens/sec, requests/sec
   - **GPU**: Utilization, memory, temperature
   - **Cost**: $/token, $/request

**Capacity Planning**:
```python
# Assumptions
daily_active_users = 1_000_000
requests_per_user_per_day = 10
total_daily_requests = daily_active_users * requests_per_user_per_day  # 10M

# Peak hour (10% of daily traffic)
peak_hour_requests = total_daily_requests * 0.1  # 1M
requests_per_second = peak_hour_requests / 3600  # 278 req/s

# With vLLM on 8× A100 (assuming 50 req/s throughput per GPU)
gpus_needed = requests_per_second / 50  # ~6 GPUs
pods_needed = gpus_needed / 8  # ~1 pod of 8 GPUs + redundancy = 2 pods

print(f"Infrastructure: 2× pods (8× A100 each)")
print(f"Peak capacity: ~800 req/s")
print(f"Cost: ~$50K/month (AWS p4d.24xlarge)")
```

**Scaling Strategy**:
- **Horizontal**: Add more inference servers
- **Vertical**: Larger GPUs (H100 > A100)
- **Model**: Smaller models for simple queries (routing)

### Q2: Design a RAG System for Enterprise Knowledge Base

**Requirements**:
- 10TB of documents (PDFs, Word, internal wikis)
- Real-time updates
- Multi-tenancy (100 organizations)
- Accuracy > 90% on retrieval
- Latency < 3s end-to-end

**Architecture** (see [10_RAG/projects/advanced_project/README.md](../../10_RAG/projects/advanced_project/README.md) for full details):

**Key decisions**:

1. **Chunking Strategy**:
   - Recursive chunking (512 tokens, 50 overlap)
   - Preserve document structure (sections, tables)
   - Store metadata (source, timestamp, tenant_id)

2. **Embeddings**:
   - Model: OpenAI text-embedding-3-large (3072-dim)
   - Batch processing: 100 docs/sec
   - Cost: ~$0.13 per 1M tokens

3. **Vector Database**:
   - Weaviate or Qdrant (both support multi-tenancy)
   - HNSW index (M=16, efConstruction=200)
   - Horizontal sharding by tenant

4. **Retrieval**:
   - **Hybrid**: Dense (vector) + Sparse (BM25)
   - **Reranking**: Cohere rerank-english-v3.0
   - **Query expansion**: HyDE (hypothetical document embeddings)

5. **Generation**:
   - GPT-4 or Claude with streaming
   - Citation tracking (chunk IDs)
   - Fact-checking against source

6. **Evaluation**:
   - RAGAS framework (faithfulness, relevance)
   - Human eval on 1000 test queries
   - A/B testing in production

**Scaling considerations**:
- **Ingestion**: Kafka queue + worker pool
- **Retrieval**: Cache embeddings for common queries
- **Multi-tenancy**: Namespace isolation + RBAC
- **Monitoring**: LangSmith, Prometheus, Grafana

## Section 4: Optimization Questions

### Q: How would you reduce LLM inference latency by 50%?

**Systematic Approach**:

#### 1. Model-Level Optimizations

| Technique | Speedup | Quality Impact | Complexity |
|-----------|---------|----------------|------------|
| **Quantization** (FP16→INT8) | 2-3x | Minimal (<1% accuracy drop) | Low |
| **Pruning** (structured) | 1.5-2x | 2-5% accuracy drop | Medium |
| **Knowledge Distillation** | 2-4x | 5-10% accuracy drop | High |
| **Layer Reduction** | 1.3-1.5x | 3-7% accuracy drop | Low |

#### 2. Kernel-Level Optimizations

- **Flash Attention**: 2-3x attention speedup
- **Fused kernels**: Combine LayerNorm + Linear
- **Custom CUDA kernels**: Optimize for specific GPU arch

#### 3. System-Level Optimizations

- **Continuous batching** (vLLM): 10-20x throughput
- **Speculative decoding**: 2-3x speedup (no quality loss)
- **KV cache optimization**: PagedAttention
- **Pipeline parallelism**: Hidden pipeline bubbles

#### 4. Application-Level Optimizations

- **Prompt caching**: Cache system prompt KV
- **Response caching**: Cache full responses
- **Smaller models for routing**: Use fast model to route to appropriate model
- **Streaming**: Start sending tokens immediately

**Recommended Stack** for 50% latency reduction:
1. FP16 quantization (2x)
2. Flash Attention (1.5x)
3. Continuous batching via vLLM (improves throughput → lower queue time)
4. Prompt caching (30% of requests have same prefix)

**Total**: ~3x speedup → 67% latency reduction ✓

## Section 5: Quick Fire Q&A

### Fine-Tuning & Training

**Q: LoRA vs Full Fine-Tuning - when to use each?**

| Aspect | LoRA | Full Fine-Tuning |
|--------|------|------------------|
| **Parameters trained** | 0.1-1% (low-rank adapters) | 100% |
| **Memory** | Low (can fit 65B on single A100) | High |
| **Speed** | 3-5x faster | Baseline |
| **Quality** | Good for task adaptation | Best for domain shift |
| **Use case** | Instruction tuning, style | New language, domain |

**Answer**: Use LoRA when you want to adapt a pre-trained model to a specific task without changing the core capabilities. Use full fine-tuning when you need to significantly change the model's knowledge (e.g., new language, specialized domain like medical/legal).

---

**Q: Explain RLHF in 30 seconds**

**Answer**: 
1. Train reward model on human preferences (thumbs up/down)
2. Use PPO to optimize LLM to maximize reward
3. Add KL penalty to prevent drift from base model

**Why**: Makes models helpful, harmless, honest (e.g., ChatGPT)

---

**Q: What's the difference between DPO and RLHF?**

**Answer**:
- **RLHF**: Separate reward model + RL training (complex, unstable)
- **DPO**: Direct optimization from preference pairs (simpler, stable)

DPO is newer (2023), easier to implement, comparable quality. Used in Zephyr, Tulu.

---

### Model Architecture

**Q: Multi-Query Attention (MQA) vs Grouped-Query Attention (GQA)?**

| Type | K,V heads | Memory | Speed | Quality |
|------|-----------|--------|-------|--------|
| Standard | n_heads | High | Slow | Best |
| **GQA** (LLaMA 2) | few groups | Medium | Fast | Good |
| **MQA** (PaLM) | 1 | Low | Fastest | Degraded |

**Answer**: GQA is the sweet spot (used in LLaMA 2). Shares K,V across groups of Q heads. Faster inference, lower KV cache memory, minimal quality loss.

---

**Q: Why are modern LLMs decoder-only?**

**Answer**:
1. **Simplicity**: Single architecture for all tasks
2. **Scalability**: Easier to scale to 100B+ params
3. **In-context learning**: Emerges naturally with scale
4. **Generation**: Most applications need generation

BERT-style encoders still used for embeddings/classification, but decoder-only dominates generative AI.

---

### Production & Serving

**Q: How do you handle rate limiting for LLM API?**

**Answer**:
- **Tokens per minute (TPM)**: Limit based on output tokens
- **Requests per minute (RPM)**: Simple rate limit
- **Cost-based**: Track $/user, set budgets
- **Priority queues**: Premium users get faster service

Implementation: Token bucket algorithm, Redis for distributed state.

---

**Q: What metrics do you monitor for LLM in production?**

**Answer**:
1. **Latency**: P50, P95, P99 (time to first token, tokens/sec)
2. **Throughput**: Requests/sec, tokens/sec
3. **Quality**: BLEU, ROUGE, human eval, thumbs up/down
4. **Cost**: $/request, $/token, GPU utilization
5. **Errors**: Timeout rate, OOM, API errors
6. **Safety**: Toxic content %, refusal rate

Use: Prometheus + Grafana for infra, LangSmith/W&B for LLM-specific.

---

This notebook covers the core LLM/System Design interview topics. Practice explaining these concepts clearly and building these systems!